# Gray-Box AIA72 — Reviewer-Ready Results Synthesis

Consolidates the completed 72 h Gray-Box evidence into manuscript-ready tables, figures, and a reviewer-safe results narrative. **No new tuning or model selection is performed.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Gray-Box AIA72 — Reviewer-Ready Results Synthesis
# ============================================================
#
# PURPOSE
# Consolidate the frozen 72 h Gray-Box evidence into manuscript-ready
# tables and figures for Prof. Rami/reviewers.
#
# IMPORTANT
# - No tuning.
# - No threshold changes.
# - No policy redesign.
# - No new model fitting.
# - Reads only already-produced frozen/post-hoc result artifacts.
#
# Outputs:
#   1) evidence_timeline.csv
#   2) clean_2026_main_results.csv
#   3) conformal_shift_summary.csv
#   4) robustness_key_scenarios.csv
#   5) manuscript_results_summary.md
#   6) figures/*.png

ROOT = Path("/home/abmoses2000")
OUT = ROOT / "graybox_aia72_results_synthesis_v2_20261005"
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
(OUT / "figures").mkdir(parents=True)

# ---------- Inputs ----------
POLICY_V1 = ROOT / "graybox_aia72_policy_validation_v1_20261005"
CYCLE25 = ROOT / "graybox_aia72_cycle25_mondrian_v2_eval_20261005"
SUPP2026 = ROOT / "graybox_aia72_supplementary2026_v3_eval_20261005"
STATS = ROOT / "graybox_aia72_supplementary2026_v3_stats_20261005"
ROB_AIA = ROOT / "graybox_aia72_posthoc_robustness_v1_20261005"
ROB_FALLBACK = ROOT / "graybox_aia72_posthoc_fallback_failure_v1_20261005"

needed = [
    POLICY_V1/"policy_results.csv",
    POLICY_V1/"conformal_policy_validation_coverage.csv",
    CYCLE25/"cycle25_policy_results.csv",
    CYCLE25/"cycle25_mondrian_coverage.csv",
    SUPP2026/"supplementary2026_policy_results.csv",
    SUPP2026/"supplementary2026_mondrian_coverage.csv",
    STATS/"point_metrics.csv",
    STATS/"bootstrap_paired_deltas.csv",
    STATS/"mcnemar_exact.csv",
    ROB_AIA/"scenario_summary.csv",
    ROB_AIA/"resilience_deltas_v3_vs_fusion.csv",
    ROB_FALLBACK/"scenario_summary.csv",
    ROB_FALLBACK/"deltas_v3_vs_fusion.csv",
]
for p in needed:
    if not p.is_file():
        raise FileNotFoundError(p)

# ---------- Load ----------
v1_results = pd.read_csv(POLICY_V1/"policy_results.csv")
v1_cov = pd.read_csv(POLICY_V1/"conformal_policy_validation_coverage.csv")

c25_results = pd.read_csv(CYCLE25/"cycle25_policy_results.csv")
c25_cov = pd.read_csv(CYCLE25/"cycle25_mondrian_coverage.csv")

s26_results = pd.read_csv(SUPP2026/"supplementary2026_policy_results.csv")
s26_cov = pd.read_csv(SUPP2026/"supplementary2026_mondrian_coverage.csv")

point = pd.read_csv(STATS/"point_metrics.csv")
boot = pd.read_csv(STATS/"bootstrap_paired_deltas.csv")
mcnemar = pd.read_csv(STATS/"mcnemar_exact.csv")

rob_aia = pd.read_csv(ROB_AIA/"scenario_summary.csv")
rob_aia_delta = pd.read_csv(ROB_AIA/"resilience_deltas_v3_vs_fusion.csv")
rob_fb = pd.read_csv(ROB_FALLBACK/"scenario_summary.csv")
rob_fb_delta = pd.read_csv(ROB_FALLBACK/"deltas_v3_vs_fusion.csv")

# ---------- 1) Evidence timeline ----------
timeline = pd.DataFrame([
    {
        "stage":"Marginal conformal v1 policy validation",
        "role":"policy_validation",
        "status":"failed operationally",
        "key_evidence":"Marginal coverage looked high overall, but flare-class coverage collapsed; q90 abstained on 209/422 flares."
    },
    {
        "stage":"Mondrian/class-conditional conformal v2",
        "role":"conformal_calibration",
        "status":"calibration repair",
        "key_evidence":"Balanced class-conditional calibration coverage near nominal 90% on reserved conformal block."
    },
    {
        "stage":"Mondrian v2 Cycle 25 evaluation",
        "role":"retrospective_cycle25",
        "status":"diagnostic failure",
        "key_evidence":"Class-conditional coverage degraded under temporal shift; q90 retained only 56.7% of flares."
    },
    {
        "stage":"Fallback-first v3 freeze",
        "role":"development redesign",
        "status":"frozen before 2026",
        "key_evidence":"Uncertainty routes to SHARP fallback; ABSTAIN reserved for fallback unavailability."
    },
    {
        "stage":"Fallback-first v3 clean 2026 evaluation",
        "role":"supplementary_2026",
        "status":"clean future evaluation",
        "key_evidence":"100% availability and flare retention; small non-significant TSS gain vs fusion, lower TSS but far fewer false alarms than SHARP."
    },
    {
        "stage":"Post-hoc robustness",
        "role":"supplementary_2026",
        "status":"post-hoc",
        "key_evidence":"AIA outages are absorbed by SHARP fallback; SHARP failure makes ABSTAIN unavoidable."
    }
])
timeline.to_csv(OUT/"evidence_timeline.csv", index=False)

# ---------- 2) Clean 2026 main result table ----------
keep_cols = [
    "policy_id","availability_rate","positive_retention_rate","abstained_flares",
    "false_alarms","issued_recall","issued_specificity","issued_precision",
    "issued_f1","issued_tss","issued_hss"
]
main_2026 = s26_results[keep_cols].copy()
main_2026.to_csv(OUT/"clean_2026_main_results.csv", index=False)

# ---------- 3) Conformal shift summary ----------
def cov_extract(df, period):
    # tolerate naming differences
    class1 = "class1_coverage" if "class1_coverage" in df.columns else "positive_coverage"
    class0 = "class0_coverage" if "class0_coverage" in df.columns else "negative_coverage"
    overall = "overall_coverage" if "overall_coverage" in df.columns else "empirical_coverage"
    out = df[["branch",overall,class0,class1]].copy()
    out.columns = ["branch","overall_coverage","no_flare_coverage","flare_coverage"]
    out["period"]=period
    return out

cov_summary = pd.concat([
    cov_extract(v1_cov,"policy_validation_marginal_v1"),
    cov_extract(c25_cov,"cycle25_mondrian_v2"),
    cov_extract(s26_cov,"supplementary2026_mondrian_v2"),
], ignore_index=True)
cov_summary.to_csv(OUT/"conformal_shift_summary.csv", index=False)

# ---------- 4) Robustness key scenarios ----------
# Pull representative rows only.
key_rows=[]

def pick(df, stype, param_col, param_val, system):
    x=df[(df["scenario_type"]==stype)&(df[param_col]==param_val)&(df["system"]==system)]
    if len(x):
        return x.iloc[0].to_dict()
    return None

for rate in [0.10,0.20,0.40]:
    for system in ["baseline_always_fusion","fallback_v3_q90"]:
        r=pick(rob_aia,"random_detected_missingness","parameter",rate,system)
        if r:
            key_rows.append(r)

# independent missingness examples
for ar,sr in [(0.20,0.10),(0.40,0.10),(0.40,0.20)]:
    x=rob_fb[
        (rob_fb["scenario_type"]=="independent_missingness") &
        np.isclose(rob_fb["aia_parameter"],ar) &
        np.isclose(rob_fb["sharp_parameter"],sr) &
        (rob_fb["system"].isin(["baseline_always_fusion","fallback_v3_q90"]))
    ]
    key_rows += x.to_dict("records")

key_rob = pd.DataFrame(key_rows)
key_rob.to_csv(OUT/"robustness_key_scenarios.csv", index=False)

# ---------- Figures ----------
# Figure 1: clean 2026 TSS/recall/specificity
plot_df = point.set_index("system").loc[
    ["baseline_always_fusion","fallback_v3_q90","baseline_always_sharp"],
    ["tss","recall","specificity"]
]
ax = plot_df.plot(kind="bar", figsize=(9,5))
ax.set_title("Supplementary 2026: locked performance comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0,1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT/"figures"/"fig1_clean_2026_performance.png", dpi=220)
plt.close()

# Figure 2: flare-class conformal coverage across periods
tmp = cov_summary.pivot(index="period",columns="branch",values="flare_coverage")
ax = tmp.plot(kind="bar", figsize=(10,5))
ax.axhline(0.90, linestyle="--", linewidth=1)
ax.set_title("Flare-class conformal coverage across evaluation periods")
ax.set_ylabel("Empirical flare-class coverage")
ax.set_ylim(0,1)
ax.set_xlabel("")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUT/"figures"/"fig2_flare_class_coverage_shift.png", dpi=220)
plt.close()

# Figure 3: AIA missingness availability
x = rob_aia[
    (rob_aia["scenario_type"]=="random_detected_missingness") &
    (rob_aia["system"].isin(["baseline_always_fusion","fallback_v3_q90"]))
].copy()
for system,g in x.groupby("system"):
    plt.plot(g["parameter"],g["availability_rate_mean"],marker="o",label=system)
plt.xlabel("AIA missingness rate")
plt.ylabel("Mean forecast availability")
plt.ylim(0,1.02)
plt.title("Operational availability under detected AIA loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"figures"/"fig3_aia_missingness_availability.png", dpi=220)
plt.close()

# Figure 4: SHARP outage availability
x = rob_fb[
    (rob_fb["scenario_type"]=="random_sharp_outage") &
    (rob_fb["system"].isin(["baseline_always_fusion","fallback_v3_q90"]))
].copy()
for system,g in x.groupby("system"):
    plt.plot(g["sharp_parameter"],g["availability_rate_mean"],marker="o",label=system)
plt.xlabel("SHARP outage rate")
plt.ylabel("Mean forecast availability")
plt.ylim(0,1.02)
plt.title("Availability when fallback itself fails")
plt.legend()
plt.tight_layout()
plt.savefig(OUT/"figures"/"fig4_sharp_outage_availability.png", dpi=220)
plt.close()

# ---------- Manuscript summary ----------
def row(system):
    return point.loc[point["system"]==system].iloc[0]

fusion=row("baseline_always_fusion")
v3=row("fallback_v3_q90")
sharp=row("baseline_always_sharp")

tss_fus = boot[(boot["metric"]=="tss")&(boot["comparison"]=="q90_minus_fusion")].iloc[0]
tss_sharp = boot[(boot["metric"]=="tss")&(boot["comparison"]=="q90_minus_sharp")].iloc[0]

mc_fus = mcnemar[mcnemar["comparison"].str.contains("fusion")].iloc[0]
mc_sharp = mcnemar[mcnemar["comparison"].str.contains("sharp")].iloc[0]

md = f"""# Gray-Box AIA72 Results Synthesis

## Frozen experimental chronology

1. **Marginal conformal v1** failed operationally because high marginal coverage masked rare-flare undercoverage and the predeclared q90 policy abstained on too many flare cases.
2. **Mondrian/class-conditional conformal v2** repaired class balance on the calibration block but did not preserve nominal flare-class coverage under later temporal shift.
3. The **Cycle 25 diagnostic** showed that using uncertainty as a hard abstention trigger disproportionately withheld flare-producing cases.
4. **Fallback-first v3** was therefore frozen before the supplementary-2026 evaluation: trusted fusion is NORMAL, SHARP fallback is DEGRADED, and ABSTAIN is reserved for genuine fallback unavailability.
5. The **supplementary-2026 evaluation** remained one-shot and locked; subsequent outage tests are explicitly post-hoc robustness analyses.

## Clean supplementary-2026 result

On 11,118 cases with 992 positives:

- Fusion: TSS {fusion.tss:.3f}, recall {fusion.recall:.3f}, specificity {fusion.specificity:.3f}.
- Fallback-first v3 q90: TSS {v3.tss:.3f}, recall {v3.recall:.3f}, specificity {v3.specificity:.3f}.
- SHARP: TSS {sharp.tss:.3f}, recall {sharp.recall:.3f}, specificity {sharp.specificity:.3f}.

Relative to fusion, v3 q90 changed TSS by {tss_fus.mean_bootstrap_delta:.4f} with 95% bootstrap CI
[{tss_fus["ci2.5"]:.4f}, {tss_fus["ci97.5"]:.4f}]. The CI includes zero, so the evidence does not support a strong TSS-superiority claim.
The exact paired McNemar p-value versus fusion was {mc_fus.exact_two_sided_p:.4g}.

Relative to SHARP, v3 q90 changed TSS by {tss_sharp.mean_bootstrap_delta:.4f} with 95% bootstrap CI
[{tss_sharp["ci2.5"]:.4f}, {tss_sharp["ci97.5"]:.4f}]. SHARP therefore retained the stronger TSS/recall operating point,
while v3 provided substantially higher specificity/precision and fewer false alarms.

## Conformal result

The central uncertainty finding is not that conformal prediction failed universally, but that guarantees calibrated on earlier data did not remain class-wise reliable under later temporal shift.
Rare-flare coverage degraded much more strongly than no-flare coverage, demonstrating why aggregate coverage alone is inadequate for this task.

## Operational robustness

Detected AIA loss can be routed to SHARP fallback, preserving forecast availability and flare retention. As AIA missingness increases,
fusion-only availability declines while v3 remains available so long as SHARP is healthy.

When SHARP itself fails, the frozen architecture loses both the multimodal fusion route and the fallback route.
In that case ABSTAIN becomes unavoidable. This provides empirical justification for the three operational states:

- **NORMAL:** trusted multimodal fusion.
- **DEGRADED:** SHARP fallback.
- **ABSTAIN:** no valid fallback path.

## Reviewer-safe contribution statement

The main contribution is not a claim of universal predictive superiority. It is an experimentally validated operational trust layer
that exposes when uncertainty-based abstention is unsafe for rare events, preserves service through modality-aware fallback,
and makes unavoidable abstention explicit when the fallback itself is unavailable.
"""
(OUT/"manuscript_results_summary.md").write_text(md)

protocol={
    "status":"GRAYBOX_AIA72_RESULTS_SYNTHESIS_COMPLETE",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "new_model_fitting":False,
    "new_threshold_tuning":False,
    "new_policy_selection":False,
    "new_policy_redesign":False,
    "purpose":"reviewer-ready synthesis of already frozen and post-hoc evidence",
    "outputs":[
        "evidence_timeline.csv",
        "clean_2026_main_results.csv",
        "conformal_shift_summary.csv",
        "robustness_key_scenarios.csv",
        "manuscript_results_summary.md",
        "figures/fig1_clean_2026_performance.png",
        "figures/fig2_flare_class_coverage_shift.png",
        "figures/fig3_aia_missingness_availability.png",
        "figures/fig4_sharp_outage_availability.png",
    ]
}
(OUT/"protocol.json").write_text(json.dumps(protocol,indent=2)+"\n")

print("===== GRAY-BOX AIA72 RESULTS SYNTHESIS COMPLETE =====")
print("new model fitting: False")
print("new threshold tuning: False")
print("new policy selection: False")
print("new policy redesign: False")
print("\nOutputs:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(" ",p)
print("\n--- Manuscript summary ---\n")
print(md)


===== GRAY-BOX AIA72 RESULTS SYNTHESIS COMPLETE =====
new model fitting: False
new threshold tuning: False
new policy selection: False
new policy redesign: False

Outputs:
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/clean_2026_main_results.csv
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/conformal_shift_summary.csv
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/evidence_timeline.csv
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/figures/fig1_clean_2026_performance.png
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/figures/fig2_flare_class_coverage_shift.png
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/figures/fig3_aia_missingness_availability.png
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/figures/fig4_sharp_outage_availability.png
  /home/abmoses2000/graybox_aia72_results_synthesis_v2_20261005/manuscript_results_summary.md
  /home/abmoses2000/graybox_aia72_r